# ZTF × MostHosts Cross-Match

This notebook merges two catalogs:

| Catalog | File | Key column |
|---|---|---|
| ZTF SN Ia photometric fits | `data/ZTF_snia_data.csv` | `ztfname` |
| MostHosts host-galaxy catalog | `data/mosthosts_20250408.csv` | `sn_name_sp` |

Rows are matched when `ztfname == sn_name_sp`.  
All ZTF columns are prefixed with `ztf_`, all MostHosts columns with `mosthost_`.  
The merged table is written to `data/ztf_mosthost_match.csv`.

> **Note:** MostHosts can list multiple candidate host galaxies per SN (`hostnum` 0, 1, 2 ...),  
> so the merged table will have more rows than the number of unique matched SNe.

## 0. Imports

In [13]:
import pandas as pd

## 1. Load the catalogs

- `index_col=0` on the ZTF file drops the unnamed row-index column that pandas wrote when the CSV was originally saved.
- `low_memory=False` silences dtype-inference warnings caused by mixed-type columns in the large MostHosts file.

In [14]:
ztf  = pd.read_csv('data/ZTF_snia_data.csv',      index_col=0, low_memory=False)
most = pd.read_csv('data/mosthosts_20250408.csv',              low_memory=False)

print(f'ZTF rows      : {len(ztf):,}  |  unique names     : {ztf["ztfname"].nunique():,}')
print(f'MostHosts rows: {len(most):,}  |  unique sn_name_sp: {most["sn_name_sp"].nunique():,}')

ZTF rows      : 3,628  |  unique names     : 3,628
MostHosts rows: 109,014  |  unique sn_name_sp: 66,465


## 2. Check overlap before merging

How many ZTF SNe have a corresponding entry in MostHosts?

In [15]:
ztf_names  = set(ztf['ztfname'])
most_names = set(most['sn_name_sp'])
overlap    = ztf_names & most_names

print(f'ZTF names found in MostHosts: {len(overlap):,} / {len(ztf_names):,}')
print(f'Sample matches              : {sorted(overlap)[:5]}')

ZTF names found in MostHosts: 725 / 3,628
Sample matches              : ['ZTF18aaadqua', 'ZTF18aaajrso', 'ZTF18aaasvcd', 'ZTF18aabasml', 'ZTF18aabqgnb']


## 3. Prefix columns and merge

Every ZTF column gets a `ztf_` prefix and every MostHosts column gets a `mosthost_` prefix  
**before** the join, so there is no column-name ambiguity in the merged output.

An **inner join** is used so only SNe present in both catalogs appear in the output.

In [16]:
# prefix all columns before joining to avoid ambiguity
ztf.columns  = ['ztf_'      + c for c in ztf.columns]
most.columns = ['mosthost_' + c for c in most.columns]

# inner join: only keep SNe present in both catalogs
merged = pd.merge(
    ztf,
    most,
    left_on  = 'ztf_ztfname',         # ZTF name in the ZTF table
    right_on = 'mosthost_sn_name_sp', # ZTF name in the MostHosts table
    how      = 'inner'
)

print(f'Merged rows : {len(merged):,}')
print(f'Columns     : {len(merged.columns)}')
merged.head(3)

Merged rows : 1,344
Columns     : 114


,ztf_ztfname,ztf_redshift,ztf_redshift_err,ztf_source,ztf_t0,ztf_x0,ztf_x1,ztf_c,ztf_t0_err,ztf_x0_err,...,mosthost_ba_leda_sga,mosthost_z_leda_sga,mosthost_ref_sga,mosthost_group_id_sga,mosthost_group_name_sga,mosthost_group_ra_sga,mosthost_group_dec_sga,mosthost_group_diameter_sga,mosthost_d26_sga,mosthost_d26_ref_sga
0,ZTF18aaadqua,0.078672,0.002807,z_snid,58130.778798,0.000845,4.999999,-0.392734,14.743977,0.000103,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ZTF18aaadqua,0.078672,0.002807,z_snid,58130.778798,0.000845,4.999999,-0.392734,14.743977,0.000103,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ZTF18aaadqua,0.078672,0.002807,z_snid,58130.778798,0.000845,4.999999,-0.392734,14.743977,0.000103,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Save output

In [17]:
out_path = 'data/ztf_mosthost_match.csv'
merged.to_csv(out_path, index=False)
print(f'Saved {len(merged):,} rows x {len(merged.columns)} columns -> {out_path}')

Saved 1,344 rows x 114 columns -> data/ztf_mosthost_match.csv


## 5. Check `mosthost_ls_id_dr9` coverage

Verify whether every row in the merged catalog has a Legacy Survey DR9 ID (`mosthost_ls_id_dr9`),
and display the value for every row.

In [20]:
# Load the merged catalog from disk (so this cell can be run independently)
merged = pd.read_csv('data/ztf_mosthost_match.csv', low_memory=False)

# --- coverage check ---
total_rows  = len(merged)
missing     = merged['mosthost_ls_id_dr9'].isna().sum()
has_id      = total_rows - missing

print(f'Total rows             : {total_rows:,}')
print(f'Rows WITH  ls_id_dr9  : {has_id:,}')
print(f'Rows WITHOUT ls_id_dr9: {missing:,}')
print(f'Coverage               : {has_id / total_rows * 100:.1f}%')
print()

# --- print ls_id_dr9 for every row ---
# Display SN name, host number, and the DR9 ID side-by-side for easy reading
id_table = merged[['ztf_ztfname','ztf_x0', 'ztf_c', 'mosthost_hostnum', 'mosthost_ls_id_dr9']].copy()
id_table.index = range(len(id_table))  # clean 0-based index
print(id_table.to_string())

Total rows             : 1,344
Rows WITH  ls_id_dr9  : 1,279
Rows WITHOUT ls_id_dr9: 65
Coverage               : 95.2%

       ztf_ztfname    ztf_x0     ztf_c  mosthost_hostnum  mosthost_ls_id_dr9
0     ZTF18aaadqua  0.000845 -0.392734                 0        9.906628e+15
1     ZTF18aaadqua  0.000845 -0.392734                 1        9.906628e+15
2     ZTF18aaadqua  0.000845 -0.392734                 2        9.906628e+15
3     ZTF18aaasvcd       NaN       NaN                 0        9.907737e+15
4     ZTF18aaasvcd       NaN       NaN                 1        9.907737e+15
5     ZTF18aabasml  0.000458  1.008026                 0        9.907734e+15
6     ZTF18aabasml  0.000458  1.008026                 1        9.907734e+15
7     ZTF18aabqgnb  0.001015  0.546846                 0        9.907734e+15
8     ZTF18aabqgnb  0.001015  0.546846                 1        9.907734e+15
9     ZTF18aabssme  0.002602  0.010830                 0        9.906627e+15
10    ZTF18aabssme  0.002602  0.0

## 6. Filter: drop rows missing `mosthost_ls_id_dr9`

Remove any row where `mosthost_ls_id_dr9` is NaN (i.e. no Legacy Survey DR9 match exists for that host)
and save the cleaned table to a new file.

In [21]:
# Load the merged catalog from disk
merged = pd.read_csv('data/ztf_mosthost_match.csv', low_memory=False)

# Drop rows where mosthost_ls_id_dr9 or ztf_x0 is NaN
filtered = merged.dropna(subset=['mosthost_ls_id_dr9', 'ztf_x0'])

# Save to new file
filtered.to_csv('data/ztf_mosthost_match_filtered_lc_ls_id.csv', index=False)

In [23]:
merged = pd.read_csv('data/ztf_mosthost_match_filtered_lc_ls_id.csv', low_memory=False)
id_table = merged[['ztf_ztfname','ztf_x0', 'ztf_c', 'mosthost_hostnum', 'mosthost_ls_id_dr9']].copy()
id_table.index = range(len(id_table))  # clean 0-based index
print(id_table.to_string())

       ztf_ztfname    ztf_x0     ztf_c  mosthost_hostnum  mosthost_ls_id_dr9
0     ZTF18aaadqua  0.000845 -0.392734                 0        9.906628e+15
1     ZTF18aaadqua  0.000845 -0.392734                 1        9.906628e+15
2     ZTF18aaadqua  0.000845 -0.392734                 2        9.906628e+15
3     ZTF18aabasml  0.000458  1.008026                 0        9.907734e+15
4     ZTF18aabasml  0.000458  1.008026                 1        9.907734e+15
5     ZTF18aabqgnb  0.001015  0.546846                 0        9.907734e+15
6     ZTF18aabqgnb  0.001015  0.546846                 1        9.907734e+15
7     ZTF18aabssme  0.002602  0.010830                 0        9.906627e+15
8     ZTF18aabssme  0.002602  0.010830                 1        9.906627e+15
9     ZTF18aabstmw  0.001256  0.310154                 0        9.907735e+15
10    ZTF18aabtaor  0.000914 -0.128799                 0        9.907736e+15
11    ZTF18aabtaor  0.000914 -0.128799                 1        9.907736e+15